# Earnings Overnight Backtest

Single-leg 2% OTM option (call or put, whichever has higher OI × Volume — "follow
smart money") entered on the trading day **before** earnings (T-1, last 30 min before
close) and exited on **earnings day** (T, at the open). Cost capped at $1,000 per
position; events whose top candidate busts the cap are dropped.

**Prerequisites** — pre-populate the data caches via the CLI:

```bash
# Options + stock for SPY (EODHD paid plan required)
just download SPY

# Earnings calendar (this package's own CLI)
just download-earnings --symbols SPY
```


In [ ]:
from dotenv import load_dotenv

load_dotenv()

from datetime import date

from options_strategies.earnings_overnight import (
    EarningsOvernightConfig,
    run_earnings_overnight,
)
from options_strategies.shared import load_earnings_overnight_data
from backtest_charts import BacktestReport

## Configuration

`as_of_date` defaults to `date.today()` so EODHD's pre-published future
events are filtered out (no peeking ahead in a point-in-time backtest).
Override to a specific past date to inspect what the strategy "knew"
at that point in time.

In [ ]:
config = EarningsOvernightConfig(
    symbol="SPY",
    capital=100_000.0,
    # Strike selection
    otm_target_pct=0.02,        # 2% OTM
    otm_tolerance_pct=0.005,    # ±0.5% strike band
    reference_price="high",     # T-1 daily high ≈ 3:30 PM
    # Liquidity filter ("follow smart money")
    min_oi=100,
    min_volume=50,
    # Expiration
    max_entry_dte=14,           # "nearest expiry" — keep it cheap
    min_entry_dte=0,
    # Cost cap
    cost_cap_usd=1_000.0,
    # Capital split between call and put legs (each is 0.5 by default;
    # only one fires per event, so the unused capital sits idle)
    call_weight=0.5,
    put_weight=0.5,
    # Point-in-time cutoff
    as_of_date=date.today(),
)
config

## Load Data

In [ ]:
print(f"Loading data for {config.symbol}…")
options, stock, earnings = load_earnings_overnight_data(
    config.symbol,
    as_of_date=config.as_of_date,
)
print(f"  Options: {len(options):,} rows")
print(f"  Stock:   {len(stock):,} rows")
print(f"  Earnings: {sum(len(v) for v in earnings.values())} events cached")
for code_, dates in earnings.items():
    print(f"    {code_}: {len(dates)} events ({dates[0].date()} → {dates[-1].date()})")

## Run Backtest

In [ ]:
print("Running earnings-overnight backtest…")
result = run_earnings_overnight(options, stock, earnings, config)

## Summary

In [ ]:
s = result.summary
print("═══ Earnings Overnight Portfolio Summary ═══")
print(f"  Total trades:    {s.get('total_trades', 0)}")
print(f"  Win rate:        {s.get('win_rate', 0):.1%}")
print(f"  Total P&L:       ${s.get('total_pnl', 0):,.2f}")
print(f"  Max drawdown:    {s.get('max_drawdown', 0):.2%}")
print(f"  Sharpe ratio:    {s.get('sharpe_ratio', 0):.2f}")
print(f"  Profit factor:   {s.get('profit_factor', 0):.2f}")
print(f"  Avg days held:   {s.get('avg_days_in_trade', 0):.1f}")

## Per-Leg Results

In [ ]:
for name, leg in result.leg_results.items():
    ls = leg.summary
    print(f"\n  ── {name} leg ──")
    print(
        f"    Trades: {ls.get('total_trades', 0)}  "
        f"Win rate: {ls.get('win_rate', 0):.1%}  "
        f"P&L: ${ls.get('total_pnl', 0):,.2f}"
    )

## Trade Log

In [ ]:
if not result.trade_log.empty:
    result.trade_log.head(20)
else:
    print("No trades — check filters / cache coverage.")

## Caveats (read these before trusting the numbers)

1. **T-1 daily `high` is a 3:30 PM proxy.** optopsy's data is EOD-only;
   the actual 3:30 PM price could be lower than the daily high. For
   SPY/QQQ the 3:00–4:00 PM range is typically < 0.1 %, so the proxy
   is tight; for names with volatile closes the strike ends up further
   OTM than intended.
2. **T EOD is a 9:30 AM open proxy.** The post-earnings exit P&L uses
   the closing bid/ask of the day earnings are announced, which is
   hours after the open. The realized volatility of an actual
   9:30 AM open → 4 PM close trade is understated here.
3. **$1,000 cost cap is strict.** If the top OI×Volume candidate's
   ask × 100 × 1 exceeds the cap, the event is dropped (no entry, no
   quantity rescaling, no fallback to the lower-score side).
4. **Single-symbol workflow.** Looping over a portfolio of symbols is
   a future extension.
5. **EODHD options history is ~2 years.** SPY/QQQ have enough
   coverage; IWM (~1.4 years) is too thin for a 2-year backtest.


## Visualizations

### Create report

In [ ]:
report = BacktestReport(result, capital=config.capital)

### Equity curve

In [ ]:
report.plot_equity()

### Cumulative P&L by leg

In [ ]:
report.plot_cum_pnl()

### Per-trade P&L distribution

In [ ]:
report.plot_pnl_dist()

### Exit type breakdown

In [ ]:
report.plot_exits()

### Strategy summary table

In [ ]:
report.plot_summary()

### Full dashboard

In [ ]:
report.plot_dashboard()